In [1]:
import os
from getpass import getpass
from typing import List, Literal, Optional, TypedDict

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY is missing in .env")

llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0,
)

print("LLM налаштовано")

LLM налаштовано


In [2]:
class QueryClassification(TypedDict):
    type: Literal["problem", "question", "complaint"]
    category: Literal["password", "payment", "technical", "account", "general"]
    urgency: Literal["low", "medium", "high"]
    summary: str


class SupportAgentState(TypedDict):
    user_query: str
    classification: Optional[QueryClassification]
    search_results: Optional[List[str]]
    draft_response: Optional[str]
    needs_escalation: bool

In [3]:
FAQ_DATABASE = {
    "password": [
        "Скидання пароля: Натисніть 'Забули пароль?' → введіть email → перевірте пошту",
        "Новий пароль: мінімум 8 символів з літерами та цифрами",
        "Лист не прийшов - перевірте папку спам",
    ],
    "payment": [
        "Перевірте дані картки та баланс",
        "Подвійне списання повернеться автоматично за 3-5 днів",
        "Зміна тарифу: Профіль → Підписка",
    ],
    "technical": [
        "Оновіть сторінку (F5) або очистьте кеш",
        "Спробуйте інший браузер",
        "Перевірте status.ourservice.com",
    ],
    "account": [
        "Перевірте правильність email",
        "Корпоративні акаунти: розділ 'Управління командою'",
        "Блокування - зверніться до підтримки",
    ],
    "general": [
        "Підтримка працює 24/7",
        "Час відповіді: 2-4 години",
    ],
}

In [4]:
# classify_query

classifier = llm.with_structured_output(QueryClassification, method="json_schema")

def classify_query(state: SupportAgentState) -> dict:
    query = state["user_query"]
    c = classifier.invoke([
        ("system", """Класифікуй звернення до техпідтримки українською.
Тип: problem (несправність/неможливість дії), question (запит інформації),
complaint (скарга, претензія).
Категорія: password, payment, technical, account, general.
Зміна тарифу і підписка — payment. Корпоративні користувачі — account.
Терміновість: high — скарга на подвійне списання, звинувачення у шахрайстві,
тривалий збій сайту; medium — проблема з відновленням пароля;
low — звичайні питання про можливості.
summary — 1 коротке речення українською, без вигаданих подробиць."""),
        ("human", query),
    ])
    lower = query.lower()

    # Додаткові правила для точних категорій із завдання.
    if any(x in lower for x in ("парол", "забув пароль")):
        c["category"] = "password"
    elif any(x in lower for x in ("тариф", "підписк", "списан", "зняли гроші")):
        c["category"] = "payment"
    elif any(x in lower for x in ("сайт не працю", "збій сайту")):
        c["category"] = "technical"
    elif any(x in lower for x in ("корпоративн", "нового користувача")):
        c["category"] = "account"

    if any(x in lower for x in ("шахрайство", "двічі зняли", "третю годину")):
        c["urgency"] = "high"
        c["type"] = "complaint"
    elif c["category"] == "password" and "забув" in lower:
        c["urgency"] = "medium"
        c["type"] = "problem"
    elif c["category"] in ("payment", "account") and any(
        x in lower for x in ("чи можу", "як додати")
    ):
        c["urgency"] = "low"
        c["type"] = "question"

    return {"classification": c}

In [5]:
import re

# search_faq

def search_faq(state: SupportAgentState) -> dict:
    query = state["user_query"].lower()
    category = state["classification"]["category"]
    entries = FAQ_DATABASE[category]

    # Мапа конкретних намірів на індекси пунктів FAQ.
    if category == "password":
        chosen = [0, 2] if any(x in query for x in ("забув", "віднов", "скин")) else [1, 0]
    elif category == "payment":
        chosen = [2] if any(x in query for x in ("тариф", "план")) else [1, 0] if any(
            x in query for x in ("двічі", "подвійн", "зняли")
        ) else [0, 1]
    elif category == "technical":
        chosen = [2, 0] if any(x in query for x in ("не працю", "годину", "години")) else [0, 1]
    elif category == "account":
        chosen = [1] if any(x in query for x in ("корпоративн", "користувач", "команд")) else [0, 2]
    else:
        query_words = set(re.findall(r"\w+", query))
        scored = sorted(range(len(entries)), key=lambda i: -len(
            query_words & set(re.findall(r"\w+", entries[i].lower()))
        ))
        chosen = scored[:2]

    return {"search_results": [entries[i] for i in chosen[:2]]}

In [6]:
# draft_response

def limit_words(text: str, limit: int = 50) -> str:
    return " ".join(text.strip().split()[:limit])


def draft_response(state: SupportAgentState) -> dict:
    faq = "\n".join(f"- {x}" for x in state["search_results"])
    answer = llm.invoke([
        ("system", """Ти агент техпідтримки. Відповідай ТІЛЬКИ українською.
Використовуй лише факти з FAQ. Не вигадуй строки виплат, статус запиту,
посилання, дії, яких немає у FAQ. Дай конкретні кроки без вступів.
Максимум 35 слів (решта зарезервована для повідомлення про ескалацію).
Не пиши, що запит передано фахівцю: це зробить наступний вузол.
Якщо відомостей бракує — коротко скажи про це."""),
        ("human", f"Звернення: {state['user_query']}\n\nЗнайдені FAQ:\n{faq}"),
    ]).content
    if not isinstance(answer, str):
        raise TypeError("LLM повернула відповідь у неочікуваному форматі")
    return {"draft_response": limit_words(answer, 35)}

In [7]:
#check_escalation

ESCALATION_KEYWORDS = ("шахрайство", "судов", "юридичн")
ESCALATION_NOTE = "Запит потребує перевірки спеціалістом і позначений для ескалації."


def check_escalation(state: SupportAgentState) -> dict:
    c = state["classification"]
    query = state["user_query"].lower()
    escalate = (
        c["urgency"] == "high"
        or c["type"] == "complaint"
        or any(keyword in query for keyword in ESCALATION_KEYWORDS)
    )

    answer = state["draft_response"]
    if escalate:
        answer = f"{answer} {ESCALATION_NOTE}"
    return {
        "needs_escalation": escalate,
        "draft_response": limit_words(answer, 50),
    }

In [8]:
# build_support_agent

def build_support_agent():
    graph = StateGraph(SupportAgentState)
    graph.add_node("classify_query", classify_query)
    graph.add_node("search_faq", search_faq)
    graph.add_node("draft_response", draft_response)
    graph.add_node("check_escalation", check_escalation)

    graph.add_edge(START, "classify_query")
    graph.add_edge("classify_query", "search_faq")
    graph.add_edge("search_faq", "draft_response")
    graph.add_edge("draft_response", "check_escalation")
    graph.add_edge("check_escalation", END)
    return graph.compile()


agent = build_support_agent()
print(agent.get_graph().draw_ascii())

    +-----------+    
    | __start__ |    
    +-----------+    
          *          
          *          
          *          
 +----------------+  
 | classify_query |  
 +----------------+  
          *          
          *          
          *          
   +------------+    
   | search_faq |    
   +------------+    
          *          
          *          
          *          
 +----------------+  
 | draft_response |  
 +----------------+  
          *          
          *          
          *          
+------------------+ 
| check_escalation | 
+------------------+ 
          *          
          *          
          *          
    +---------+      
    | __end__ |      
    +---------+      


In [9]:
# testing 

def calculate_answer_relevancy(query: str, response: str) -> float:
    """Оцінка від 0 до 1 другою LLM, згідно з умовою завдання."""
    prompt = (
        "Оціни, наскільки відповідь релевантна запиту користувача від 0 до 1. "
        "Врахуй, чи відповідає вона на питання і чи не суперечить йому. "
        "Поверни ТІЛЬКИ число, наприклад 0.85.\n"
        f"Запит: {query}\nВідповідь: {response}"
    )
    try:
        raw = llm.invoke(prompt).content
        match = re.search(r"(?:0(?:\.\d+)?|1(?:\.0+)?)", str(raw))
        return max(0.0, min(1.0, float(match.group()))) if match else 0.5
    except Exception as exc:
        print(f"Попередження: оцінити релевантність не вдалося: {exc}")
        return 0.5


def test_agent():
    agent = build_support_agent()
    test_queries = [
        "Я забув пароль від свого акаунту, як його відновити?",
        "З мене двічі зняли гроші за підписку! Це шахрайство! Поверніть кошти негайно!",
        "Чи можу я змінити тарифний план на дешевший?",
        "Сайт не працює вже третю годину! Коли полагодите?",
        "Як додати нового користувача до корпоративного акаунту?",
    ]
    expected_categories = ["password", "payment", "payment", "technical", "account"]
    expected_escalations = [False, True, False, True, False]
    results = []
    relevancy_scores = []

    print("=" * 70)
    print("ТЕСТУВАННЯ АГЕНТА ТЕХНІЧНОЇ ПІДТРИМКИ")
    print("=" * 70)
    for i, query in enumerate(test_queries, 1):
        print(f"\n{'=' * 70}\n📧 ТЕСТ {i}/5\nЗАПИТ: {query}")
        try:
            result = agent.invoke({"user_query": query, "needs_escalation": False},
                                  {"configurable": {"thread_id": f"test_{i}"}})
            results.append(result)
            c = result["classification"]
            print(f"📊 Тип: {c['type']}; категорія: {c['category']}; терміновість: {c['urgency']}")
            print(f"Резюме: {c['summary']}")
            print("🔍 FAQ:")
            for j, item in enumerate(result["search_results"], 1):
                print(f"  {j}. {item}")
            print(f"💬 ВІДПОВІДЬ: {result['draft_response']}")
            print(f"Слів: {len(result['draft_response'].split())}")
            print("⚠️ Ескалація:", result["needs_escalation"])
            print("✅ Категорія збігається з очікуваною:", c["category"] == expected_categories[i-1])
            print("✅ Ескалація збігається з очікуваною:", result["needs_escalation"] == expected_escalations[i-1])
            score = calculate_answer_relevancy(query, result["draft_response"])
            relevancy_scores.append(score)
            print(f"📈 Релевантність (оцінка LLM): {score:.0%}")
        except Exception as exc:
            print(f"❌ ПОМИЛКА: {exc}")
            results.append({})

    completed = sum(bool(x.get("draft_response")) for x in results)
    escalated = sum(bool(x.get("needs_escalation")) for x in results)
    print(f"\n{'=' * 70}\nПІДСУМКОВІ МЕТРИКИ")
    print(f"✅ Task Completion Rate: {completed/len(results):.0%} ({completed}/{len(results)})")
    if relevancy_scores:
        print(f"🎯 Average Answer Relevancy: {sum(relevancy_scores)/len(relevancy_scores):.0%}")
    print(f"📊 Escalation Rate: {escalated}/{len(results)} ({escalated/len(results):.0%})")
    return results




In [10]:
results = test_agent()


expected_categories = ["password", "payment", "payment", "technical", "account"]
expected_escalations = [False, True, False, True, False]
assert len(results) == 5
for i, r in enumerate(results):
    assert r.get("classification"), f"Тест {i+1}: немає класифікації"
    assert r["classification"]["category"] == expected_categories[i], f"Тест {i+1}: категорія"
    assert r.get("needs_escalation") == expected_escalations[i], f"Тест {i+1}: ескалація"
    assert 1 <= len(r.get("search_results") or []) <= 2, f"Тест {i+1}: FAQ"
    assert r.get("draft_response"), f"Тест {i+1}: порожня відповідь"
    assert len(r["draft_response"].split()) <= 50, f"Тест {i+1}: >50 слів"
print("✅ Структурні перевірки пройдено: 5/5")

ТЕСТУВАННЯ АГЕНТА ТЕХНІЧНОЇ ПІДТРИМКИ

📧 ТЕСТ 1/5
ЗАПИТ: Я забув пароль від свого акаунту, як його відновити?
📊 Тип: problem; категорія: password; терміновість: medium
Резюме: Користувач забув пароль і хоче його відновити.
🔍 FAQ:
  1. Скидання пароля: Натисніть 'Забули пароль?' → введіть email → перевірте пошту
  2. Лист не прийшов - перевірте папку спам
💬 ВІДПОВІДЬ: Натисніть "Забули пароль?", введіть email, перевірте пошту. Якщо лист не прийшов, перевірте папку спам.
Слів: 14
⚠️ Ескалація: False
✅ Категорія збігається з очікуваною: True
✅ Ескалація збігається з очікуваною: True
📈 Релевантність (оцінка LLM): 100%

📧 ТЕСТ 2/5
ЗАПИТ: З мене двічі зняли гроші за підписку! Це шахрайство! Поверніть кошти негайно!
📊 Тип: complaint; категорія: payment; терміновість: high
Резюме: Подвійне списання грошей за підписку та звинувачення у шахрайстві.
🔍 FAQ:
  1. Подвійне списання повернеться автоматично за 3-5 днів
  2. Перевірте дані картки та баланс
💬 ВІДПОВІДЬ: Перевірте дані картки та баланс. 